In [15]:
from pathlib import Path
import os 

PROJECT_ROOT = Path.cwd()

sound=PROJECT_ROOT/"sample_voice_note.mp3"
out_path=PROJECT_ROOT/"spoken_summary.mp3"

print(sound)

c:\Users\Lenovo\GenAI-Agentic-Systems\LLM-Foundations-Prompt-Engineering\MultiModel-Pipelines\sample_voice_note.mp3


In [3]:
from groq import Groq

from dotenv import load_dotenv
load_dotenv()

api_key=os.getenv("api_key")
client=Groq(api_key=api_key)


In [12]:
from gtts import gTTS

def speech_to_text(path):
    with open(path,"rb") as audio_file:
        result=client.audio.transcriptions.create(
            file=audio_file,
            model="whisper-large-v3",
            response_format="text"
        )
    return result.strip() if isinstance(result, str) else str(result).strip()

def summarizer(transcript):
    prompt = (
        "Summarize in exactly 3 short bullet points. "
        "Keep only useful facts. Do not invent details.\n\n"
        f"Transcript:\n{transcript}"
    )
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,
    )
    return response.choices[0].message.content.strip()

def text_to_speech(summary:str,outfile:Path):
    speakable=summary.replace("-"," ").replace("/n",". ")
    tts=gTTS(text=speakable,lang="en")
    tts.save(str(outfile))
    return outfile




    

In [16]:
# --- Execute the Pipeline ---
print("🚀 Running Speech Pipeline...\n")

transcript = speech_to_text(sound)
print("🎙️ TRANSCRIPT:")
print(transcript, "\n")

summary = summarizer(transcript)
print("📝 SUMMARY:")
print(summary, "\n")

out_audio = text_to_speech(summary, out_path)
print(f"🔊 SUCCESS: Saved spoken summary to -> {out_audio}")

🚀 Running Speech Pipeline...

🎙️ TRANSCRIPT:
Hello, this is a short campus update. The library is open until 10 pm. Please carry your student ID. 

📝 SUMMARY:
Here are 3 short bullet points summarizing the transcript:

- The library is open until 10 pm.
- The library is located on a campus.
- Students need to carry their student ID. 

🔊 SUCCESS: Saved spoken summary to -> c:\Users\Lenovo\GenAI-Agentic-Systems\LLM-Foundations-Prompt-Engineering\MultiModel-Pipelines\spoken_summary.mp3
